# MDTF ESNB Notebook: North Atlantic Ocean POD

Runs the `natl_ocean` POD (Maroon et al., `emaroon/cmip_input_noamoc` branch) through the ESNB interface on CESM2 historical r1i1p1f1.

Validated end-to-end 2026-04-22 with:
- 2 yrs (1980–1981) of CESM2 monthly ocean data from `/glade/collections/cmip/CMIP6/…`
- Obs: `/glade/campaign/cgd/ccr/yeager/Sub2Sub/POD_data/obs_1x1.nc` (symlinked into `inputdata/obs_data/natl_ocean/`)

**Kernel:** `Python (esnb)` (set via the top-right kernel selector).

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, copy, json
from pathlib import Path
import xarray as xr

os.environ['CODE_ROOT'] = '/glade/work/taydral/MDTF/MDTF-diagnostics'
os.environ['WORK_DIR']  = '/glade/work/taydral/MDTF/wkdir/MDTF_output/natl_ocean'
os.environ['OBS_DATA']  = '/glade/work/taydral/MDTF/inputdata/obs_data/natl_ocean/'
Path(os.environ['WORK_DIR']).mkdir(parents=True, exist_ok=True)

module_path = os.environ['CODE_ROOT']
if module_path not in sys.path:
    sys.path.append(module_path)

from src.util import json_utils

## Section 1: POD Settings

Load from `diagnostics/natl_ocean/settings.jsonc`. We also build a monthly-only copy for ESNB — it can't time-slice the fx vars (`areacello`, `volcello`), so we load those manually in Section 3.

In [2]:
settings_full = json_utils.read_json(
    os.path.join(module_path, 'diagnostics/natl_ocean/settings.jsonc')
)

settings_monthly = copy.deepcopy(settings_full)
for fx_var in ('areacello', 'volcello'):
    settings_monthly['varlist'].pop(fx_var, None)

print('varlist (ESNB):', list(settings_monthly['varlist'].keys()))
print('varlist (full):', list(settings_full['varlist'].keys()))

varlist (ESNB): ['tos', 'vsf', 'hfds', 'so', 'thetao']
varlist (full): ['tos', 'vsf', 'hfds', 'so', 'thetao', 'volcello', 'areacello']


In [3]:
startdate, enddate = '1980-01-01', '1981-12-31'

case_info = {
    'case_list': {
        'CESM2_historical_r1i1p1f1': {
            'model': 'CESM', 'convention': 'CMIP',
            'startdate': startdate, 'enddate': enddate,
        }
    },
    'DATA_CATALOG':   str(Path(module_path) / 'diagnostics/natl_ocean/CMIP_CESM_historical_001.json'),
    'OBS_DATA_ROOT':  '/glade/work/taydral/MDTF/inputdata/obs_data',
    'WORK_DIR':       os.environ['WORK_DIR'],
    'OUTPUT_DIR':     os.environ['WORK_DIR'],
    'conda_root':     '/glade/u/home/taydral/miniconda3',
    'conda_env_root': '/glade/u/home/taydral/miniconda3/envs',
    'micromamba_exe': '', 'large_file': False,
    'make_multicase_figure_html': False, 'make_variab_tar': False,
    'overwrite': True, 'pod_list': ['natl_ocean'],
    'run_pp': True, 'save_pp_data': True, 'save_ps': False,
    'translate_data': True, 'user_pp_scripts': [''],
}
print(f"Catalog: {case_info['DATA_CATALOG']} (exists: {Path(case_info['DATA_CATALOG']).exists()})")

Catalog: /glade/work/taydral/MDTF/MDTF-diagnostics/diagnostics/natl_ocean/CMIP_CESM_historical_001.json (exists: True)


## Section 2: Case / Runtime Config

## Section 3: Load Data (ESNB monthly + manual fx merge + obs)

In [4]:
from esnb import NotebookDiagnostic, CaseGroup2

pod_env_vars = NotebookDiagnostic(settings_monthly)
groups = [CaseGroup2(case_info, date_range=(startdate, enddate))]
pod_env_vars.resolve(groups)
pod_env_vars.open()
print(f'ESNB loaded {len(pod_env_vars.datasets)} monthly datasets')

# Manual load for the fx (time-invariant) vars
fx_base = Path('/glade/collections/cmip/CMIP6/CMIP/NCAR/CESM2/historical/r1i1p1f1/Ofx')
ds_areacello = xr.open_dataset(fx_base / 'areacello/gn/v20190308/areacello_Ofx_CESM2_historical_r1i1p1f1_gn.nc')
ds_volcello  = xr.open_dataset(fx_base / 'volcello/gn/v20190308/volcello_Ofx_CESM2_historical_r1i1p1f1_gn.nc')

ds_merged = xr.merge(pod_env_vars.datasets + [ds_areacello, ds_volcello], join='outer')
print(f"\nmerged vars: {sorted(list(ds_merged.data_vars))}")
print(f"merged dims: {dict(ds_merged.sizes)}")

/glade/u/home/taydral/miniconda3/envs/esnb/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'tos' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/glade/u/home/taydral/miniconda3/envs/esnb/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'vsf' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/glade/u/home/taydral/miniconda3/envs/esnb/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'hfds' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/glade/u/home/taydral/miniconda3/envs/esnb/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'so' has multiple fill values {np.float32(1e+20

ESNB loaded 5 monthly datasets


/glade/u/home/taydral/miniconda3/envs/esnb/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'areacello' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/glade/u/home/taydral/miniconda3/envs/esnb/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'volcello' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)



merged vars: ['areacello', 'hfds', 'lat_bnds', 'lev_bnds', 'lon_bnds', 'so', 'thetao', 'tos', 'volcello', 'vsf']
merged dims: {'nlat': 384, 'nlon': 320, 'time': 24, 'lev': 60, 'vertices': 4, 'd2': 2}


/glade/derecho/scratch/taydral/tmp/ipykernel_89305/2066145702.py:14: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge(pod_env_vars.datasets + [ds_areacello, ds_volcello], join='outer')
/glade/derecho/scratch/taydral/tmp/ipykernel_89305/2066145702.py:14: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.mer

- Dictionary merge 
- keep obs in separate dictionary 

In [5]:
ds_obs = xr.open_dataset('/glade/work/taydral/MDTF/inputdata/obs_data/natl_ocean/obs_1x1.nc').load()
print(f'obs vars: {list(ds_obs.data_vars)}')
print(f'obs dims: {dict(ds_obs.sizes)}')

obs vars: ['thetao_zavg', 'so_zavg', 'sigma0_zavg', 'sic', 'mld']
obs dims: {'month': 12, 'lat': 180, 'lon': 360}


## Section 4: POD diagnostics (call into `POD_utils`)

Helper functions from `diagnostics/natl_ocean/POD_utils.py`:
`compute_sigma0`, `compute_mld`, `compute_zavg`, `regrid`, `SpatialPlot_climo_bias`, `error_stats`, …

In [6]:
pod_dir = Path(module_path) / 'diagnostics/natl_ocean'
if str(pod_dir) not in sys.path:
    sys.path.append(str(pod_dir))

import POD_utils
fns = [f for f in dir(POD_utils) if not f.startswith('_') and callable(getattr(POD_utils, f))]
print('POD_utils functions:', fns[:15])

POD_utils functions: ['BoundaryNorm', 'GridSpec', 'LinearSegmentedColormap', 'ScatterPlot_Error', 'Scatter_panel', 'SpatialBias_panel', 'SpatialPlot_climo_bias', 'SpatialRank_panel', 'blue2red_cmap', 'calc_dens', 'calc_maps', 'calc_wmt', 'check_depth_units', 'compute_mld', 'compute_sigma0']


### 4.1 Smoke test — compute σ₀ on a small CESM slice

In [7]:
tiny = ds_merged.isel(time=0, lev=slice(0, 5), nlat=slice(300, 320), nlon=slice(50, 70))
sigma0 = POD_utils.compute_sigma0(tiny['thetao'], tiny['so'])
print(f'sigma0 shape {sigma0.shape}, mean {float(sigma0.mean()):.3f} kg/m^3')

sigma0 shape (5, 20, 20), mean 19.839 kg/m^3


### 4.2 Full workflow (to be populated as we iterate)

Mirror of `natl_driver.py`:
1. `compute_sigma0(thetao, so)` → σ₀
2. `compute_mld(sigma0)` → MLD
3. `compute_zavg(ds, var, dz, depth=200)` → upper-200m thickness-weighted means
4. `regrid(ds, method='bilinear')` → 1°×1°
5. monthly climatology via `groupby('time.month').mean('time')`
6. `SpatialPlot_climo_bias(ds_target, ds_model, ds_obs, var, ...)` → bias maps

In [ ]:
# TODO: wire up full climatology / bias / plot chain